In [ ]:


from generate_recipe_task import *


In [ ]:
recipe_df = pd.read_csv("../csv/full_dataset.csv", index_col=0)
recipe_df.head()

In [ ]:
num_ings = [len(eval(row.loc['NER'])) for i, row in recipe_df.iterrows()]
num_steps = [len(eval(row.loc['directions'])) for i, row in recipe_df.iterrows()]
recipe_df['num_steps'] = num_steps
recipe_df['num_ings'] = num_ings

In [ ]:
ing_bins = [0, 8, 16, 32, 64, 128]
step_bins = [0, 8, 16, 32] #64, 128]

stepwise_bins_map = {}

for i in range(1, len(ing_bins)):
    for j in range(1, len(step_bins)):
        print(f'Current Bin: ing={ing_bins[i]}, step={step_bins[j]}')
        stepwise_bins_map[(ing_bins[i], step_bins[j])] = recipe_df[ing_bins[i] >= recipe_df['num_ings']][ing_bins[i-1] < recipe_df['num_ings']][step_bins[j] >= recipe_df['num_steps']][step_bins[j-1] < recipe_df['num_steps']]
        print(f'Recipes in current bin: {len(stepwise_bins_map[(ing_bins[i], step_bins[j])])}')
        print()

print(len(stepwise_bins_map))

In [ ]:
for ing, step in stepwise_bins_map:
    cur = stepwise_bins_map[(ing, step)]
    cur.sample(min(50, len(cur))).to_csv(f'recipes_by_bin/s{step}i{ing}.csv')

# Transcript Processing

In [ ]:


import numpy as np

from generate_recipe_task import *

import os

def process(files):
    for file in files:
        df = pd.read_csv(f'recipes_by_bin/{file}', index_col=0)

        print(f'Current Bin: {file}')
        print(f'Avg ingredients: {'%.3f'%np.mean(df['num_ings'])}')
        print(f'Avg steps: {'%.3f'%np.mean(df['num_steps'])}')
        print()
        print('Generating transcripts:')

        transcripts = []
        for i in tqdm(range(len(df))):
            row = df.iloc[i]
            if not isinstance(row['transcript'], float):
                transcripts.append(row['transcript'])
                continue
            rt = None

            try: rt = generate_ingredient_transcript(eval(row['ingredients']), eval(row['directions']), use_openai=True)
            except Exception as e: print(f'Exception {type(e)} for {row["title"]}')

            transcripts.append(rt)
        df['transcript'] = transcripts
        # print(df.columns)
        df.to_csv(f'recipes_by_bin/{file}')

        print()
        print('='*50)
        print()

In [ ]:
files = os.listdir('../recipes_by_bin')
# files.sort()
s8 = [file for file in files if file.startswith('s8')]
s8.sort(key=lambda s: len(s))
s16 = [file for file in files if file.startswith('s16')]
s16.sort(key=lambda s: len(s))
s32 = [file for file in files if file.startswith('s32')]
s32.sort(key=lambda s: len(s))

In [ ]:
process(s8)

In [ ]:
process(s16)

In [ ]:
process(s32)
for file in files:
    df = pd.read_csv(f'recipes_by_bin/{file}', index_col=0)

In [ ]:
full = pd.read_csv('../csv/stepwise_transcripts.csv', index_col=0)
for file in os.listdir('../recipes_by_bin'):
    df = pd.read_csv(f'recipes_by_bin/{file}', index_col=0)
    if len(df) > 0: full = pd.concat((full, df))

full = full.loc[~full.index.duplicated(keep='first'), :]
full.to_csv(f'csv/stepwise_transcripts.csv')
full

# Ingredient Variables:
- ### \# of ingredients mentioned in transcript
- ### \# of ingredients tested
- ### \# of ingredients in recipe total

# Step Variables:
- ### \# of steps mentioned in transcript (transcript length)
- ### \# of steps in the recipe total

In [1]:

import os
from tqdm import tqdm
from generate_recipe_task import *
from testing_old.experiment_classes import WrongIngTest

stepwise_transcripts = pd.read_csv('./csv/stepwise_transcripts.csv')

together_list = [
        "deepseek-ai/DeepSeek-R1",
        "OpenAI/gpt-oss-20B",
        "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
        "Qwen/Qwen3.5-397B-A17B",
        # "moonshotai/Kimi-K2-Instruct-0905",
    ]
openai_list = [
        'gpt-5',
        'gpt-5-mini',
    ]
testing_bins = [2,4,8,16]

Testing parameters:

4,2 4,4 4,8 4,16

2,4 4,4 8,4 16,4

dimensions:
- 6 models
- 8 parameter versions
- 50+ recipes

Organization:

per model results
- parameter csv
    - recipe transcript, used/unused, recipe examples, results

In [2]:
def valid_rows(df, params, actor_specs):
    rows = []
    for i, row in df.iterrows():
        try:
            transcript, used, unused = transcript_to_task(eval(row['transcript']), eval(row['ingredients']), actor_specs, params)
            # print(len(transcript.split('\n')), len(used), len(unused))
            if (len(transcript.split('\n')) >= params['transcript_length'] and
                    len(used) >= params['transcript_ings'] and
                    len(unused) >= params['unused_ings']):
                rows.append(i)
        except Exception as e: pass # print(f'Exception {type(e)} for {row["title"]}')
    return rows

In [ ]:
for model in openai_list:
    model_name = model.split('/')[-1]

    try: os.mkdir(f'results/{model_name}')
    except OSError as e: pass

    full = pd.read_csv('../csv/stepwise_transcripts.csv', index_col=0)

    def test_on(axis):
        for bin in testing_bins:
            if os.path.exists(f'results/{model}/{axis}={bin}.csv'): continue

            params = {'transcript_ings': 4, 'unused_ings': 4, 'num_examples': 4, 'transcript_length': 4, axis: bin}
            metadata = {
                'id':[],
                'title': [],
                'transcript': [],
                'transcript_text': [],
                'used': [],
                'unused': [],
                'examples': [],
            }
            results = {
                'id':[],
                'outputs': [],
                'full_outputs': [],
                'correct': [],
                'overall_acc': []
            }

            actor_specs = ["Alice", "She", 1, 0.5, 0]

            valid = valid_rows(full, params, actor_specs)

            print(f'Testing {axis}={bin}:')
            for i in tqdm(range(min(len(valid), 50))):
                id = valid[i]
                row = full.loc[[id]]
                metadata['id'].append(id)
                metadata['title'].append(row['title'].item())
                transcript_text, used, unused = transcript_to_task(eval(row['transcript'].item()), eval(row['ingredients'].item()), actor_specs, params)
                metadata['transcript'].append(eval(row['transcript'].item()))
                metadata['transcript_text'].append(transcript_text)
                metadata['used'].append(used)
                metadata['unused'].append(unused)
                test = WrongIngTest(transcript_text, used, unused, actor_specs, 0)
                metadata['examples'].append(test.perturb_recipe(max_examples=params['num_examples']))

                results['id'].append(id)
                outputs, full_outputs = test.run_test(use_openai=True, model=model, n=1)

                accuracies = test.get_results(match_exact=False)

                results['outputs'].append(outputs)
                results['full_outputs'].append(full_outputs)
                results['correct'].append(accuracies)
                results['overall_acc'].append(sum(accuracies)/len(accuracies))


            metadata_df = pd.DataFrame(metadata)
            metadata_df.to_csv(f'results/metadata/{axis}={bin}.csv', index=False)

            results_df = pd.DataFrame(results)
            results_df.to_csv(f'results/{model}/{axis}={bin}.csv', index=False)

    # vary transcript ingredients
    test_on('transcript_ings')
    test_on('unused_ings')



In [ ]:
for model in together_list:
    print(f'Testing model {model}:')

    model_name = model.split('/')[-1]

    try: os.mkdir(f'results/{model_name}')
    except OSError as e: pass

    full = pd.read_csv('../csv/stepwise_transcripts.csv', index_col=0)

    def test_on(axis):
        for bin in testing_bins:
            if os.path.exists(f'results/{model_name}/{axis}={bin}.csv'): continue

            params = {'transcript_ings': 4, 'unused_ings': 4, 'num_examples': 4, 'transcript_length': 4, axis: bin}
            metadata = {
                'id':[],
                'title': [],
                'transcript': [],
                'transcript_text': [],
                'used': [],
                'unused': [],
                'examples': [],
            }
            results = {
                'id':[],
                'outputs': [],
                'full_outputs': [],
                'correct': [],
                'overall_acc': []
            }

            actor_specs = ["Alice", "She", 1, 0.5, 0]

            valid = valid_rows(full, params, actor_specs)

            print(f'Testing {axis}={bin}:')
            for i in tqdm(range(min(len(valid), 50))):
                id = valid[i]
                row = full.loc[[id]]
                metadata['id'].append(id)
                metadata['title'].append(row['title'].item())
                transcript_text, used, unused = transcript_to_task(eval(row['transcript'].item()), eval(row['ingredients'].item()), actor_specs, params)
                metadata['transcript'].append(eval(row['transcript'].item()))
                metadata['transcript_text'].append(transcript_text)
                metadata['used'].append(used)
                metadata['unused'].append(unused)
                test = WrongIngTest(transcript_text, used, unused, actor_specs, 0)
                metadata['examples'].append(test.perturb_recipe(max_examples=params['num_examples']))

                results['id'].append(id)
                outputs, full_outputs = test.run_test(use_openai=False, model=model, n=1)

                accuracies = test.get_results(match_exact=False)

                results['outputs'].append(outputs)
                results['full_outputs'].append(full_outputs)
                results['correct'].append(accuracies)
                results['overall_acc'].append(sum(accuracies)/len(accuracies))


            metadata_df = pd.DataFrame(metadata)
            metadata_df.to_csv(f'results/metadata/{axis}={bin}.csv', index=False)

            results_df = pd.DataFrame(results)
            results_df.to_csv(f'results/{model_name}/{axis}={bin}.csv', index=False)

    # vary transcript ingredients
    test_on('unused_ings')
    test_on('transcript_ings')



In [ ]:
client = Together()

## Uploads batch job file
file_resp = client.files.upload(
    file="../results_old/DeepSeek-R1/transcript_ings=2.jsonl", purpose="batch-api", check=False
)

In [ ]:
file_id = file_resp.id

batch = client.batches.create(
    input_file_id=file_id, endpoint="/v1/chat/completions"
)

print(batch.job.id)

In [ ]:
batch_stat = client.batches.retrieve(batch.job.id)

print(batch_stat.status)

# Testing Example Generation

In [3]:
model = 'gpt-5'
print(f'Testing model {model}:')

model_name = model.split('/')[-1]

try: os.mkdir(f'results/{model_name}')
except OSError as e: pass

full = pd.read_csv('../csv/stepwise_transcripts.csv', index_col=0)

params = {'transcript_ings': 4, 'unused_ings': 2, 'num_examples': 4, 'transcript_length': 4}

metadata = {
    'id':[],
    'title': [],
    'transcript': [],
    'transcript_text': [],
    'used': [],
    'unused': [],
    'examples': [],
}
results = {
    'id':[],
    'outputs': [],
    'full_outputs': [],
    'correct': [],
    'overall_acc': []
}

actor_specs = ["Alice", "She", 1, 0.5, 0]

valid = valid_rows(full, params, actor_specs)

valid_long = []
for id in valid:
    row = full.loc[[id]]
    transcript_text, used, unused = transcript_to_task(eval(row['transcript'].item()), eval(row['ingredients'].item()), actor_specs, params)

    extra_ings = eval(row['ingredients'].item())
    try:
        for ing in used: extra_ings.remove(ing)
        for ing in unused: extra_ings.remove(ing)
    except: continue
    extra_ings = [ing.split(', ')[0].split('or')[0] for ing in extra_ings]
    if len(extra_ings) < params['num_examples'] - params['unused_ings']: continue
    valid_long.append(id)

print(f'Long valid: {len(valid_long)}')

for i in tqdm(range(min(len(valid_long), 50))):
    id = valid_long[i]
    row = full.loc[[id]]
    metadata['id'].append(id)
    metadata['title'].append(row['title'].item())
    transcript_text, used, unused = transcript_to_task(eval(row['transcript'].item()), eval(row['ingredients'].item()), actor_specs, params)

    extra_ings = eval(row['ingredients'].item())
    for ing in used: extra_ings.remove(ing)
    for ing in unused: extra_ings.remove(ing)
    extra_ings = [ing.split(', ')[0].split('or')[0] for ing in extra_ings]
    # print(extra_ings)
    if len(extra_ings) < params['num_examples'] - params['unused_ings']: continue

    unused = [ing.split(', ')[0].split('or')[0] for ing in unused]
    # print(unused)

    metadata['transcript'].append(eval(row['transcript'].item()))
    metadata['transcript_text'].append(transcript_text)
    metadata['used'].append(used)
    metadata['unused'].append(unused)

    test = WrongIngTest(transcript_text, used, unused, actor_specs, 0, extra_ings=extra_ings)
    # print(test.perturb_recipe(max_examples=params['num_examples']))
    # break
    metadata['examples'].append(test.perturb_recipe(max_examples=params['num_examples']))

    results['id'].append(id)
    outputs, full_outputs = test.run_test(model=model, n=1)

    accuracies = test.get_results(match_exact=False)

    results['outputs'].append(outputs)
    results['full_outputs'].append(full_outputs)
    results['correct'].append(accuracies)
    results['overall_acc'].append(sum(accuracies)/len(accuracies))


metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv(f'results/metadata/extra_ing_test.csv', index=False)

results_df = pd.DataFrame(results)
results_df.to_csv(f'results/{model_name}/extra_ing_test.csv', index=False)


Testing model gpt-5:
Long valid: 185


100%|██████████| 50/50 [19:54<00:00, 23.90s/it]


In [4]:
model = 'gpt-5'
print(f'Testing model {model}:')

model_name = model.split('/')[-1]

try: os.mkdir(f'results/{model_name}')
except OSError as e: pass

full = pd.read_csv('../csv/stepwise_transcripts.csv', index_col=0)

params = {'transcript_ings': 4, 'unused_ings': 2, 'num_examples': 4, 'transcript_length': 4}

metadata = {
    'id':[],
    'title': [],
    'transcript': [],
    'transcript_text': [],
    'used': [],
    'unused': [],
    'examples': [],
}
results = {
    'id':[],
    'outputs': [],
    'full_outputs': [],
    'correct': [],
    'overall_acc': []
}

actor_specs = ["Alice", "She", 1, 0.5, 0]

valid = valid_rows(full, params, actor_specs)

for i in tqdm(range(min(len(valid), 50))):
    id = valid[i]
    row = full.loc[[id]]
    metadata['id'].append(id)
    metadata['title'].append(row['title'].item())
    transcript_text, used, unused = transcript_to_task(eval(row['transcript'].item()), eval(row['ingredients'].item()), actor_specs, params)

    # unused = [ing.split(', ')[0].split('(')[0].split('or')[0] for ing in unused]
    # print(unused)

    metadata['transcript'].append(eval(row['transcript'].item()))
    metadata['transcript_text'].append(transcript_text)
    metadata['used'].append(used)
    metadata['unused'].append(unused)

    test = WrongIngTest(transcript_text, used, unused, actor_specs, 0, extra_ings=None)
    # print(test.perturb_recipe(max_examples=params['num_examples']))
    # break
    metadata['examples'].append(test.perturb_recipe(max_examples=params['num_examples']))

    results['id'].append(id)
    outputs, full_outputs = test.run_test(use_openai=True, model=model, n=1)

    accuracies = test.get_results(match_exact=False)

    results['outputs'].append(outputs)
    results['full_outputs'].append(full_outputs)
    results['correct'].append(accuracies)
    results['overall_acc'].append(sum(accuracies)/len(accuracies))


metadata_df = pd.DataFrame(metadata)
metadata_df.to_csv(f'results/metadata/generate_ing_test.csv', index=False)

results_df = pd.DataFrame(results)
results_df.to_csv(f'results/{model_name}/generate_ing_test.csv', index=False)


Testing model gpt-5:


100%|██████████| 50/50 [34:50<00:00, 41.82s/it]


# Old Test Code

In [ ]:
def run_test(params, actor_specs, row):
    transcript, used, unused = transcript_to_task(eval(row['transcript']), eval(row['ingredients']), actor_specs, params)

    together_list = [
        "OpenAI/gpt-oss-20B",
        "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
        "Qwen/Qwen3-235B-A22B-Instruct-2507-tput",
        # "moonshotai/Kimi-K2-Instruct-0905",
        "deepseek-ai/DeepSeek-R1",
    ]
    openai_list = [
        'gpt-5',
        'gpt-5-mini',
    ]

    results = {
        'model': [],
        'outputs': [],
        'full_outputs': [],
        'accuracy per ingredient': [],
        'overall accuracy': [],
    }

    test = WrongIngTest(transcript, used, unused, actor_specs, 0)
    print(test.perturb_recipe(False))

    for model in together_list:
        print(f'Testing model {model}: ')
        outputs, full_outputs = test.run_test(client=Together(), model=model)
        accuracies = test.get_results()
        results['model'].append(model)
        results['outputs'].append(outputs)
        results['full_outputs'].append(full_outputs)
        results['accuracy per ingredient'].append(accuracies)
        results['overall accuracy'].append(sum(accuracies)/len(accuracies))

    for model in openai_list:
        print(f'Testing model {model}: ')
        outputs, full_outputs = test.run_test(client=OpenAI(), model=model)
        accuracies = test.get_results()
        results['model'].append(model)
        results['outputs'].append(outputs)
        results['full_outputs'].append(full_outputs)
        results['accuracy per ingredient'].append(accuracies)
        results['overall accuracy'].append(sum(accuracies)/len(accuracies))

    return transcript, used, unused, results



In [ ]:
import os

params = {
    'transcript_ings': 4,
    'unused_ings': 4,
    'num_examples': 4,
    'transcript_length': 5,
}
actor_specs = ["Alice", "She", 1, 0.5, 0]

# print(len(eval(row['transcript'])))
# for ings, action, product in eval(row['transcript']):
#     print(ings)

# valid_rows = [16, 22, 25, 27, 28, 29, 32, 33, 35, 37, 38]
valid_rows = [29, 32, 33, 35, 37, 38]

for i in valid_rows:
    row = stepwise_transcripts.iloc[i]

    try: os.mkdir(f'results/{row.iloc[0]}')
    except OSError as e: pass

    r1 = list(stepwise_transcripts.columns)
    r1[0] = 'index'
    r2 = list(row)
    metadata = pd.DataFrame(r2, r1)
    metadata.to_csv(f'results/{row.iloc[0]}/metadata.csv', header=False)

    inputs = {
        'transcript_ings': [],
        'unused_ings': [],
        'transcript_length': [],
        'transcript': [],
        'used_list':[],
        'unused_list':[],
    }

    for t in [2,4,8]:
        for u in [2,4,8]:
            params = {
                'transcript_ings': t,
                'unused_ings': u,
                'transcript_length': 5,
            }
            transcript, used, unused, results = run_test(params, actor_specs, row)

            inputs['transcript_ings'].append(t)
            inputs['unused_ings'].append(u)
            inputs['transcript_length'].append(5)
            inputs['transcript'].append(transcript)
            inputs['used_list'].append(used)
            inputs['unused_list'].append(unused)

            outputs = pd.DataFrame(results)
            outputs.to_csv(f'results/{row.iloc[0]}/outputs-t{t}u{u}.csv', index=False, header=True)

    inputs = pd.DataFrame(inputs)
    print(inputs.columns)
    inputs.to_csv(f'results/{row.iloc[0]}/inputs.csv', index=False, header=True)

# TODO:
- test one var at a time, n=1 trials
- increase reasoning tokens for together models
- split unused_ings/# recipes